# TechCorner Mobile Sales Analysis
**A comprehensive analysis of 10 months of mobile sales data from TechCorner, a retail shop in Bangladesh.**

### Table of Contents
1. [Setup & Data Loading](#1-setup)
2. [Data Cleaning](#2-cleaning)
3. [Monthly Sales Trend](#3-monthly-sales)
4. [Sales by Location](#4-location)
5. [Age Distribution](#5-age)
6. [Gender Distribution](#6-gender)
7. [Top 10 Best-Selling Models](#7-top-models)
8. [Price Distribution](#8-price)
9. [Revenue Analysis](#9-revenue)
10. [Facebook Marketing Effectiveness](#10-facebook)
11. [New vs Returning Customers](#11-returning)
12. [Cross-Analyses](#12-cross)
13. [Conclusions & Recommendations](#13-conclusions)


## 1. Setup & Data Loading <a id='1-setup'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
df = pd.read_csv('TechCorner_Sales_update.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 2. Data Cleaning <a id='2-cleaning'></a>

In [ ]:
# Check missing values
print('Missing values per column:')
print(df.isnull().sum())


In [ ]:
# Convert types
df['Sell Price'] = pd.to_numeric(df['Sell Price'], errors='coerce')
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# Drop empty trailing column and nulls
df.drop(columns=['Unnamed: 11'], inplace=True, errors='ignore')
df.dropna(inplace=True)

# Rename messy columns to clean names
df.rename(columns={
    'Cus.ID': 'customer_id',
    'Cus. Location': 'location',
    'Mobile Name': 'mobile_name',
    'Sell Price': 'sell_price',
    'Does he/she Come from Facebook Page?': 'facebook_referral',
    'Does he/she Followed Our Page?': 'follows_page',
    'Did he/she buy any mobile before?': 'returning_customer',
    'Did he/she hear of our shop before?': 'heard_of_shop'
}, inplace=True)

# Feature engineering
df['month'] = df['Date'].dt.to_period('M')
df['brand'] = df['mobile_name'].str.split().str[0]
df['age_group'] = pd.cut(df['Age'], bins=[17,25,35,45,50],
                         labels=['18-25','26-35','36-45','46-50'])

print('Clean dataset info:')
df.info()

## 3. Monthly Sales Trend <a id='3-monthly-sales'></a>

In [ ]:
monthly_sales = df.groupby('month').size()
print(monthly_sales)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
monthly_sales.plot(kind='line', marker='o', color='steelblue', linewidth=2.5, ax=ax)
ax.set_title('Monthly Sales Trend', fontsize=15, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Sales')
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print('Insight: Sales grew steadily from 153 in May 2024, peaking at 943 in Oct 2024, '
      'before tapering in early 2025 — a typical post-holiday pattern.')

## 4. Sales by Customer Location <a id='4-location'></a>

In [ ]:
print(df['location'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=df, x='location', order=df['location'].value_counts().index,
              palette='coolwarm', ax=ax)
ax.set_title('Sales by Customer Location', fontsize=14, fontweight='bold')
ax.set_xlabel('Location')
ax.set_ylabel('Number of Sales')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print('Insight: Demand is nearly equal across all three zones, showing strong broad reach. '
      'Outside Rangamati leads slightly — expanding delivery options could capture even more.')

## 5. Age Distribution of Customers <a id='5-age'></a>

In [ ]:
print(df['Age'].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(df['Age'], bins=20, kde=True, color='skyblue', ax=ax)
ax.set_title('Age Distribution of Customers', fontsize=14, fontweight='bold')
ax.set_xlabel('Age')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('Insight: Average buyer age is 34. The core customer base is 26–42. '
      'Marketing efforts should prioritise this age bracket.')

## 6. Gender Distribution <a id='6-gender'></a>

In [ ]:
print(df['Gender'].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
df['Gender'].value_counts().plot.pie(autopct='%1.1f%%',
    colors=['lightblue','lightcoral'], ax=ax)
ax.set_title('Gender Distribution of Buyers', fontsize=14, fontweight='bold')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

print('Insight: Sales are almost perfectly split (50.2% F / 49.8% M). '
      'Marketing campaigns should target both genders equally.')

## 7. Top 10 Best-Selling Models <a id='7-top-models'></a>

In [ ]:
top_10 = df['mobile_name'].value_counts().nlargest(10)
print(top_10)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=top_10.index, y=top_10.values, palette='coolwarm', ax=ax)
ax.set_title('Top 10 Most Sold Mobile Models', fontsize=14, fontweight='bold')
ax.set_xlabel('Model')
ax.set_ylabel('Units Sold')
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print('Insight: Budget 5G models (Moto G85, Galaxy M35) dominate volume. '
      'Premium models (Pixel 8 Pro, iPhone 16 Pro) also appear — showing demand across price segments.')

## 8. Price Distribution <a id='8-price'></a>

In [ ]:
print(df['sell_price'].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(x=df['sell_price'], color='lightcoral', ax=ax)
ax.set_title('Sell Price Distribution (BDT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sell Price (BDT)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print('Insight: Median price is BDT 21,682. The long right tail confirms premium phone sales. '
      'Stock strategy should cover both the BDT 17K–26K sweet spot and high-end models.')

## 9. Revenue Analysis <a id='9-revenue'></a>
> **New section** — analysing total revenue, monthly revenue trend, and top brands by revenue.

In [ ]:
total_revenue = df['sell_price'].sum()
avg_revenue   = df['sell_price'].mean()
total_tx      = len(df)

print(f'Total Revenue:         BDT {total_revenue:>15,.0f}')
print(f'Total Transactions:    {total_tx:>15,}')
print(f'Avg Revenue per Sale:  BDT {avg_revenue:>15,.0f}')

In [ ]:
monthly_revenue = df.groupby('month')['sell_price'].sum()

fig, ax = plt.subplots(figsize=(11, 5))
monthly_revenue.plot(kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('Monthly Revenue (BDT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Total Revenue (BDT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(monthly_revenue.apply(lambda x: f'BDT {x:,.0f}'))

In [ ]:
brand_revenue = df.groupby('brand')['sell_price'].sum().sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=brand_revenue.index, y=brand_revenue.values, palette='viridis', ax=ax)
ax.set_title('Top 10 Brands by Total Revenue (BDT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Brand')
ax.set_ylabel('Total Revenue (BDT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=40, ha='right')
plt.tight_layout()
plt.show()

print('Insight: Samsung leads in revenue despite sharing top-10 volume spots with budget brands. '
      'This confirms that stocking premium Samsung models has a strong revenue impact.')

In [ ]:
# Estimated profit — no cost column in dataset, using 12% retail margin assumption
MARGIN = 0.12
df['est_profit'] = df['sell_price'] * MARGIN
monthly_profit = df.groupby('month')['est_profit'].sum()

total_profit = df['est_profit'].sum()
print(f'Estimated Total Profit (12% margin): BDT {total_profit:,.0f}')

fig, ax = plt.subplots(figsize=(11, 5))
monthly_profit.plot(kind='line', marker='o', color='green', linewidth=2.5, ax=ax)
ax.set_title('Estimated Monthly Profit — 12% Margin (BDT)', fontsize=14, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Estimated Profit (BDT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.2f}M'))
ax.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

print('Note: Profit margin is estimated at 12% — a typical figure for mobile retail in Bangladesh. '
      'Replace MARGIN with the actual margin when available.')

## 10. Facebook Marketing Effectiveness <a id='10-facebook'></a>

In [ ]:
print(df['facebook_referral'].value_counts())
print()
fb_pct = df['facebook_referral'].value_counts(normalize=True)*100
print(fb_pct.round(1))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.countplot(data=df, x='facebook_referral', palette='muted', ax=axes[0])
axes[0].set_title('Came from Facebook?', fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')

sns.countplot(data=df, x='follows_page', palette=['#3B9C9C','#F08080'], ax=axes[1])
axes[1].set_title('Follows Facebook Page?', fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('Count')

plt.suptitle('Facebook Marketing Effectiveness', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Insight: 34.8% of customers came via Facebook. '
      'This is a meaningful channel worth continued investment.')

## 11. New vs Returning Customers <a id='11-returning'></a>

In [ ]:
print(df['returning_customer'].value_counts())
print()
print(df['returning_customer'].value_counts(normalize=True).mul(100).round(1))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=df, x='returning_customer', palette='pastel', ax=ax)
ax.set_title('New vs Returning Customers', fontsize=13, fontweight='bold')
ax.set_xlabel('Returning Customer?')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print('Insight: 75.3% of buyers are first-time customers — strong new customer acquisition. '
      'A loyalty programme could convert more of them into repeat buyers.')

## 12. Cross-Analyses <a id='12-cross'></a>
> **New section** — deeper analysis combining variables to find actionable patterns.

### 12a. Do Facebook Customers Spend More?

In [ ]:
fb_spend = df.groupby('facebook_referral')['sell_price'].mean().round(0)
print('Average spend by referral source:')
print(fb_spend)

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x=fb_spend.index, y=fb_spend.values, palette=['#F08080','#3B9C9C'], ax=ax)
ax.set_title('Avg Spend: Facebook vs Non-Facebook Customers', fontsize=13, fontweight='bold')
ax.set_xlabel('Came from Facebook?')
ax.set_ylabel('Avg Sell Price (BDT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print('Insight: Facebook customers spend slightly more on average (BDT 25,206 vs 24,995). '
      'Facebook marketing attracts quality buyers, not just bargain hunters.')

### 12b. Which Age Group Spends the Most?

In [ ]:
age_spend = df.groupby('age_group', observed=True)['sell_price'].mean().round(0)
print('Average spend by age group:')
print(age_spend)

fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=age_spend.index, y=age_spend.values, palette='Blues_d', ax=ax)
ax.set_title('Avg Spend by Age Group', fontsize=13, fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Avg Sell Price (BDT)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

print('Insight: The 46–50 age group spends the most on average. '
      'Older customers tend to buy higher-end models — premium promotions should target them.')

### 12c. Do Returning Customers Prefer Different Brands?

In [ ]:
top_brands = df['brand'].value_counts().nlargest(6).index
df_top = df[df['brand'].isin(top_brands)]

brand_return = df_top.groupby(['brand','returning_customer']).size().unstack(fill_value=0)
brand_return_pct = brand_return.div(brand_return.sum(axis=1), axis=0) * 100
print(brand_return_pct.round(1))

brand_return_pct.plot(kind='bar', figsize=(10, 5), color=['#AED6F1','#E59866'], edgecolor='white')
plt.title('New vs Returning Customers by Brand', fontsize=13, fontweight='bold')
plt.xlabel('Brand')
plt.ylabel('% of Customers')
plt.legend(['First-time','Returning'], title='Customer Type')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print('Insight: Returning customer rates are broadly consistent across brands. '
      'Brand loyalty programmes could be applied universally.')

## 13. Conclusions & Recommendations <a id='13-conclusions'></a>

### Key Findings

| # | Finding | Implication |
|---|---------|-------------|
| 1 | **Total revenue: BDT 222M** across 8,871 transactions | Strong business performance over 10 months |
| 2 | **Sales grew steadily** from 153 (May) to 943 (Oct 2024) | Marketing and demand momentum is building |
| 3 | **Samsung leads revenue** despite sharing volume with budget brands | Premium Samsung stock drives disproportionate revenue |
| 4 | **Budget 5G models dominate volume** (Moto G85, Galaxy M35) | Midrange 5G is the primary customer demand |
| 5 | **34.8% of customers come from Facebook** | Facebook is a proven acquisition channel |
| 6 | **Facebook customers spend slightly more** (BDT 25,206 vs 24,995) | FB marketing attracts quality buyers |
| 7 | **75.3% are first-time customers** | Strong acquisition but retention needs attention |
| 8 | **Older customers (46–50) spend the most** | Premium promotions should target older demographics |
| 9 | **Demand is equal across all 3 zones** | Delivery expansion could unlock untapped outside-Rangamati demand |

---

### Recommendations

1. **Stock** — Maintain strong inventory of budget 5G models (BDT 17K–26K range) as the core volume driver. Always have the top 10 models available.
2. **Premium push** — Introduce targeted promotions for high-end Samsung, iPhone, and Pixel models aimed at the 36–50 age group.
3. **Facebook marketing** — Continue and increase Facebook investment. The channel delivers 34.8% of customers who spend slightly above average.
4. **Loyalty programme** — With 75.3% first-time buyers, a simple repeat-purchase incentive (e.g., discount on next phone) could significantly improve retention.
5. **Delivery expansion** — Outside Rangamati customers represent the largest single group. A delivery or courier partnership would capture more of this demand.
6. **Track cost price** — Adding a purchase price column to the dataset would enable true profit margin analysis rather than estimates.
